In [ ]:
link github : https://github.com/daitran1309/bai-tap-tri-tue-nhan-tao

In [1]:
# ============================================================
#   8-PUZZLE SOLVER - Thuật toán IDS (Iterative Deepening Search)
#   Sát với pseudocode trên slide
#   Dành cho người mới học, đơn giản dễ hiểu
# ============================================================

GOAL = [1, 2, 3,
        4, 5, 6,
        7, 8, 0]


# ============================================================
# Tạo NODE - lưu state + đường đi + độ sâu
# ============================================================

def tao_node(state, duong_di, do_sau):
    return {
        "state":    state,
        "duong_di": duong_di,
        "do_sau":   do_sau
    }


# ============================================================
# Mở rộng node - EXPAND(problem, node)
# Trả về danh sách các node con
# ============================================================

def EXPAND(node):
    state = node["state"]
    vi_tri_trong = state.index(0)
    hang = vi_tri_trong // 3
    cot  = vi_tri_trong % 3

    cac_con = []
    huong = [
        (hang - 1, cot),   # lên
        (hang + 1, cot),   # xuống
        (hang,     cot - 1),  # trái
        (hang,     cot + 1),  # phải
    ]

    for hang_moi, cot_moi in huong:
        if 0 <= hang_moi < 3 and 0 <= cot_moi < 3:
            vi_tri_moi = hang_moi * 3 + cot_moi
            state_moi = state.copy()
            state_moi[vi_tri_trong] = state_moi[vi_tri_moi]
            state_moi[vi_tri_moi]  = 0
            child = tao_node(state_moi, node["duong_di"] + [state_moi], node["do_sau"] + 1)
            cac_con.append(child)

    return cac_con


# ============================================================
# Kiểm tra vòng lặp - IS-CYCLE(node)
# Nếu state hiện tại đã xuất hiện trong đường đi trước đó → là chu trình
# ============================================================

def IS_CYCLE(node):
    state_hien_tai = node["state"]
    duong_di_truoc = node["duong_di"][:-1]   # bỏ state cuối (chính nó)
    return state_hien_tai in duong_di_truoc


# ============================================================
# HÀM 2: DEPTH-LIMITED-SEARCH(problem, l)
# Sát pseudocode slide:
#   frontier ← LIFO queue (stack) with NODE(problem.INITIAL)
#   result ← failure
#   while not IS-EMPTY(frontier) do
#       node ← POP(frontier)
#       if problem.IS-GOAL(node.STATE) then return node
#       if DEPTH(node) > l then result ← cutoff
#       else if not IS-CYCLE(node) do
#           for each child in EXPAND(problem, node) do
#               add child to frontier
#   return result
# ============================================================

FAILURE = "failure"
CUTOFF  = "cutoff"

def DEPTH_LIMITED_SEARCH(trang_thai_dau, trang_thai_dich, l, dem_node):
    print(f"  [DLS] Chạy với giới hạn độ sâu l = {l}")

    # frontier ← LIFO queue (stack) with NODE(problem.INITIAL)
    node_dau = tao_node(trang_thai_dau, [trang_thai_dau], do_sau=0)
    frontier = [node_dau]        # STACK - thêm cuối, lấy cuối

    # result ← failure
    result = FAILURE

    # while not IS-EMPTY(frontier) do
    while len(frontier) > 0:

        # node ← POP(frontier)
        node = frontier.pop()    # lấy từ CUỐI stack (LIFO)
        dem_node[0] += 1

        # if problem.IS-GOAL(node.STATE) then return node
        if node["state"] == trang_thai_dich:
            return node          # trả về node (mang theo duong_di)

        # if DEPTH(node) > l then result ← cutoff
        if node["do_sau"] > l:
            result = CUTOFF

        # else if not IS-CYCLE(node) do
        elif not IS_CYCLE(node):

            # for each child in EXPAND(problem, node) do
            for child in EXPAND(node):

                # add child to frontier
                frontier.append(child)

    # return result
    return result


# ============================================================
# HÀM 1: ITERATIVE-DEEPENING-SEARCH(problem)
# Sát pseudocode slide:
#   for depth = 0 to ∞ do
#       result ← DEPTH-LIMITED-SEARCH(problem, depth)
#       if result ≠ cutoff then return result
# ============================================================

def ITERATIVE_DEEPENING_SEARCH(trang_thai_dau, trang_thai_dich):
    print("=" * 50)
    print("BẮT ĐẦU ITERATIVE-DEEPENING-SEARCH (IDS)")
    print("=" * 50)

    dem_node = [0]   # đếm tổng số node đã xét (dùng list để truyền tham chiếu)

    # for depth = 0 to ∞ do
    depth = 0
    while True:

        # result ← DEPTH-LIMITED-SEARCH(problem, depth)
        result = DEPTH_LIMITED_SEARCH(trang_thai_dau, trang_thai_dich, depth, dem_node)

        # if result ≠ cutoff then return result
        if result == CUTOFF:
            print(f"  → Cutoff! Tăng độ sâu lên {depth + 1}\n")
            depth += 1
            continue

        if result == FAILURE:
            print("THẤT BẠI: Không có lời giải!")
            return None

        # result là một NODE → tìm thấy lời giải!
        print(f"\nTÌM THẤY LỜI GIẢI!")
        print(f"Độ sâu tìm thấy: {depth}")
        print(f"Số bước di chuyển: {len(result['duong_di']) - 1}")
        print(f"Tổng nodes đã xét: {dem_node[0]}\n")
        return result["duong_di"]


# ============================================================
# In bảng 3x3
# ============================================================

def in_bang(state):
    for i in range(3):
        hang = state[i*3 : i*3+3]
        hang_str = [" _" if so == 0 else f" {so}" for so in hang]
        print(" ".join(hang_str))
    print()


# ============================================================
# GIAO DIỆN TEXT - In kết quả đẹp
# ============================================================

def in_giao_dien(trang_thai_dau, loi_giai):
    print("=" * 50)
    print("LỜI GIẢI TỪNG BƯỚC:")
    print("=" * 50)
    for i, state in enumerate(loi_giai):
        label = "Ban đầu" if i == 0 else f"Bước {i}"
        print(f"{label}:")
        in_bang(state)

    print("=" * 50)
    print("SO SÁNH 3 THUẬT TOÁN:")
    print("=" * 50)
    print("""
  BFS  → QUEUE (FIFO), .pop(0) → Rộng → Ngắn nhất, tốn RAM
  DFS  → STACK (LIFO), .pop()  → Sâu  → Không tối ưu, ít RAM
  IDS  → DLS lặp tăng dần độ sâu
         = BFS (tối ưu) + DFS (ít RAM)
         Mỗi vòng lặp depth d: chạy DLS(d)
         Nếu cutoff → tăng d, chạy lại từ đầu
    """)


# ============================================================
# CHẠY CHƯƠNG TRÌNH
# ============================================================

if __name__ == "__main__":

    trang_thai_dau = [1, 2, 3,
                      4, 0, 6,
                      7, 5, 8]

    print("Trạng thái ban đầu:")
    in_bang(trang_thai_dau)
    print("Trạng thái đích:")
    in_bang(GOAL)

    loi_giai = ITERATIVE_DEEPENING_SEARCH(trang_thai_dau, GOAL)

    if loi_giai:
        in_giao_dien(trang_thai_dau, loi_giai)

Trạng thái ban đầu:
 1  2  3
 4  _  6
 7  5  8

Trạng thái đích:
 1  2  3
 4  5  6
 7  8  _

BẮT ĐẦU ITERATIVE-DEEPENING-SEARCH (IDS)
  [DLS] Chạy với giới hạn độ sâu l = 0
  → Cutoff! Tăng độ sâu lên 1

  [DLS] Chạy với giới hạn độ sâu l = 1

TÌM THẤY LỜI GIẢI!
Độ sâu tìm thấy: 1
Số bước di chuyển: 2
Tổng nodes đã xét: 16

LỜI GIẢI TỪNG BƯỚC:
Ban đầu:
 1  2  3
 4  _  6
 7  5  8

Bước 1:
 1  2  3
 4  5  6
 7  _  8

Bước 2:
 1  2  3
 4  5  6
 7  8  _

SO SÁNH 3 THUẬT TOÁN:

  BFS  → QUEUE (FIFO), .pop(0) → Rộng → Ngắn nhất, tốn RAM
  DFS  → STACK (LIFO), .pop()  → Sâu  → Không tối ưu, ít RAM
  IDS  → DLS lặp tăng dần độ sâu
         = BFS (tối ưu) + DFS (ít RAM)
         Mỗi vòng lặp depth d: chạy DLS(d)
         Nếu cutoff → tăng d, chạy lại từ đầu
    
